## PHASE 6: MODEL BUILDING & CLASSIFICATION
**Duration:** 2-3 weeks | **Deliverable:** trained_models/, model_predictions.csv

### Step 6.1: Train Multiple Classifiers

```python
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Model 1: Random Forest (good for mixed feature types)
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    class_weight='balanced',  # Penalize errors on minority class
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_balanced, y_train_balanced)

# Model 2: XGBoost (often superior)
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=7,
    learning_rate=0.1,
    scale_pos_weight=19,  # Ratio of negatives to positives
    random_state=42
)
xgb_model.fit(X_train_balanced, y_train_balanced)

# Model 3: Logistic Regression (baseline)
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_scaled, y_train_balanced)

# Store all models
models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'Logistic Regression': lr_model
}
```

### Step 6.2: Make Predictions

```python
predictions = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
    else:
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]
    
    predictions[name] = {
        'predictions': preds,
        'probabilities': probs
    }
```

---

## PHASE 7: EVALUATION & METRICS
**Duration:** 1 week | **Deliverable:** evaluation_report.xlsx, model_comparison.csv

### Step 7.1: Compute Comprehensive Metrics

```python
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    f1_score,
    roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt

for name, pred_dict in predictions.items():
    y_pred = pred_dict['predictions']
    y_pred_proba = pred_dict['probabilities']
    
    print(f"\n{'='*60}")
    print(f"MODEL: {name}")
    print(f"{'='*60}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"\nConfusion Matrix:")
    print(f"  True Negatives:  {tn:5d} (correctly identified legitimate)")
    print(f"  False Positives: {fp:5d} (legitimate flagged as fraud)")
    print(f"  False Negatives: {fn:5d} (fraud missed) ⚠️ CRITICAL")
    print(f"  True Positives:  {tp:5d} (correctly identified fraud)")
    
    # Key Metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"\nKey Metrics:")
    print(f"  Precision: {precision:.4f} (of detected fraud, how many are real)")
    print(f"  Recall:    {recall:.4f} (of actual fraud, how many did we catch) ⚠️ PRIORITY")
    print(f"  F1-Score:  {f1:.4f} (balance between precision & recall)")
    print(f"  AUC-ROC:   {auc:.4f}")
    
    # Classification Report
    print(f"\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent']))
```

### Step 7.2: Feature Importance Analysis

```python
# For Random Forest
all_features = X_train_balanced.columns.tolist()
feature_importance = pd.DataFrame({
    'feature': all_features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features (Random Forest):")
print(feature_importance.head(20).to_string())

# Highlight: are mined red flags in the top features?
red_flag_importance = feature_importance[
    feature_importance['feature'].isin(red_flag_features)
].head(10)
print("\nRed Flag Feature Importance:")
print(red_flag_importance.to_string())
```

### Step 7.3: Impact Analysis - Did Mining Help?

**Compare models with and without mined features:**

```python
# Train Random Forest WITHOUT red flag features
features_without_mining = [f for f in all_features if f not in red_flag_features]
X_train_no_mining = X_train_balanced[features_without_mining]
X_test_no_mining = X_test[features_without_mining]

rf_no_mining = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight='balanced',
    random_state=42
)
rf_no_mining.fit(X_train_no_mining, y_train_balanced)

# Compare
from sklearn.metrics import f1_score
f1_with_mining = f1_score(y_test, rf_model.predict(X_test))
f1_without_mining = f1_score(y_test, rf_no_mining.predict(X_test_no_mining))

print(f"\nMining Impact on Model Performance:")
print(f"  F1-Score WITH mined features:    {f1_with_mining:.4f}")
print(f"  F1-Score WITHOUT mined features: {f1_without_mining:.4f}")
print(f"  Improvement: {(f1_with_mining - f1_without_mining):.4f} ({(f1_with_mining/f1_without_mining - 1)*100:.1f}%)")
```

In [27]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
import pandas as pd


In [28]:

X_train_balanced = pd.read_csv("../data/splits/X_train_balanced.csv")
y_train_balanced = pd.read_csv("../data/splits/y_train_balanced.csv").values.ravel()


In [29]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_balanced, y_train_balanced)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

In [30]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=7,
    learning_rate=0.1,
    scale_pos_weight=19,
    random_state=42
)
xgb_model.fit(X_train_balanced, y_train_balanced)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [31]:
X_train_scaled=pd.read_csv("../data/splits/X_train_scaled.csv")
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_scaled, y_train_balanced)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [32]:
models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'Logistic Regression': lr_model
}

In [33]:
X_test=pd.read_csv("../data/splits/X_test.csv")
X_test_scaled=pd.read_csv("../data/splits/X_test_scaled.csv")
predictions = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
    else:
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]
    
    predictions[name] = {
        'predictions': preds,
        'probabilities': probs
    }

In [34]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    f1_score,
    roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt

In [35]:
y_test=pd.read_csv("../data/splits/y_test.csv")
for name, pred_dict in predictions.items():
    y_pred = pred_dict['predictions']
    y_pred_proba = pred_dict['probabilities']
    
    print(f"\n{'='*60}")
    print(f"MODEL: {name}")
    print(f"{'='*60}")
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"\nConfusion Matrix:")
    print(f"  True Negatives:  {tn:5d}")
    print(f"  False Positives: {fp:5d}")
    print(f"  False Negatives: {fn:5d}")
    print(f"  True Positives:  {tp:5d}")
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"\nMetrics:")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent']))


MODEL: Random Forest

Confusion Matrix:
  True Negatives:   3311
  False Positives:    27
  False Negatives:    47
  True Positives:    122

Metrics:
  Precision: 0.8188
  Recall:    0.7219
  F1-Score:  0.7673
  AUC-ROC:   0.9838

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.99      0.99      0.99      3338
  Fraudulent       0.82      0.72      0.77       169

    accuracy                           0.98      3507
   macro avg       0.90      0.86      0.88      3507
weighted avg       0.98      0.98      0.98      3507


MODEL: XGBoost

Confusion Matrix:
  True Negatives:   3242
  False Positives:    96
  False Negatives:    22
  True Positives:    147

Metrics:
  Precision: 0.6049
  Recall:    0.8698
  F1-Score:  0.7136
  AUC-ROC:   0.9873

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.99      0.97      0.98      3338
  Fraudulent       0.60      0.87      0.71       169

    a

In [36]:
feature_importance = pd.DataFrame({
    'feature': X_train_balanced.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Features:")
print(feature_importance.head(20).to_string())

Top 20 Features:
                feature  importance
1      has_company_logo    0.070215
365    has_missing_logo    0.036714
228           tfidf_per    0.023858
2         has_questions    0.022341
134       tfidf_growing    0.020230
86         tfidf_duties    0.015031
7              function    0.013947
345           tfidf_web    0.012923
317          tfidf_team    0.011312
176          tfidf_love    0.010799
369      red_flag_count    0.010740
27       tfidf_benefits    0.010485
54       tfidf_computer    0.010117
5    required_education    0.010105
240      tfidf_position    0.010035
204        tfidf_needed    0.010005
6              industry    0.009694
366    has_vague_salary    0.008587
350          tfidf_work    0.007860
50      tfidf_companies    0.007773


In [37]:
# Define red flag features used for analysis
red_flag_features = [
    'has_wire_transfer',
    'has_upfront_payment',
    'has_unverified_company',
    'has_generic_location',
    'has_urgency_language',
    'has_too_good_promise',
    'has_data_entry_money',
    'has_missing_logo',
    'has_vague_salary',
    'has_poor_grammar',
    'red_flag_count',
    'has_multiple_red_flags',
    'fraud_risk_score',
]

In [38]:
red_flag_importance = feature_importance[
    feature_importance['feature'].isin(red_flag_features)
].head(10)
print("Red Flag Features:")
print(red_flag_importance.to_string())

Red Flag Features:
                    feature  importance
365        has_missing_logo    0.036714
369          red_flag_count    0.010740
366        has_vague_salary    0.008587
360  has_unverified_company    0.007609
370  has_multiple_red_flags    0.004227
362    has_urgency_language    0.001428
367        has_poor_grammar    0.000481
361    has_generic_location    0.000349
359     has_upfront_payment    0.000055
363    has_too_good_promise    0.000036


In [39]:
all_features = X_train_balanced.columns.tolist()
features_without_mining = [f for f in all_features if f not in red_flag_features]
X_train_no_mining = X_train_balanced[features_without_mining]
X_test_no_mining = X_test[features_without_mining]

rf_no_mining = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight='balanced',
    random_state=42
)
rf_no_mining.fit(X_train_no_mining, y_train_balanced)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [40]:
f1_with_mining = f1_score(y_test, rf_model.predict(X_test))
f1_without_mining = f1_score(y_test, rf_no_mining.predict(X_test_no_mining))

print(f"F1 with mined features:    {f1_with_mining:.4f}")
print(f"F1 without mined features: {f1_without_mining:.4f}")
print(f"Improvement: {(f1_with_mining - f1_without_mining):.4f} ({(f1_with_mining/f1_without_mining - 1)*100:.1f}%)")

F1 with mined features:    0.7673
F1 without mined features: 0.7859
Improvement: -0.0186 (-2.4%)


In [41]:
import joblib

joblib.dump(rf_model, '../models/random_forest.pkl')
joblib.dump(xgb_model, '../models/xgboost.pkl')
joblib.dump(lr_model, '../models/logistic_regression.pkl')

['../models/logistic_regression.pkl']